# Model Comparison
Comparison between WOA-MF, PSO-MF, Manual baseline, and Grid Search.

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import os
import sys
import itertools
from sklearn.model_selection import train_test_split

# Set up paths relative to this notebook
NOTEBOOK_DIR = os.getcwd()
PROJECT_DIR = os.path.dirname(NOTEBOOK_DIR)
DATA_PATH = os.path.join(PROJECT_DIR, 'data', 'ml-1m', 'ratings.dat')
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
WO_RESULTS_PATH = os.path.join(PROJECT_DIR, 'models', 'woa_results.json')
FINAL_RESULTS_PATH = os.path.join(PROJECT_DIR, 'results', 'model_results.csv')

sys.path.append(SRC_DIR)

from mf import MatrixFactorization as MF
from metrics import rmse
from pso import ParticleSwarmOptimization

In [ ]:
print(f"Loading data from: {DATA_PATH}")
ratings = pd.read_csv(
    DATA_PATH,
    sep="::",
    engine="python",
    names=["user","item","rating","timestamp"]
)
ratings = ratings[["user","item","rating"]]

user_map = {u:i for i,u in enumerate(ratings.user.unique())}
item_map = {i:j for j,i in enumerate(ratings.item.unique())}

ratings["user"] = ratings.user.map(user_map)
ratings["item"] = ratings.item.map(item_map)

train, test = train_test_split(ratings.values, test_size=0.2, random_state=42)

n_users = len(user_map)
n_items = len(item_map)

In [ ]:
results_list = []

def fitness_function(params):
    lr, reg, k, epochs = params
    
    lr = np.clip(lr, 0.001, 0.02)
    reg = np.clip(reg, 0.01, 0.2)
    k = int(np.clip(k, 20, 80))
    epochs = int(np.clip(epochs, 5, 20)) 
    
    model = MF(n_users, n_items, k, lr, reg, epochs)
    model.train(train)
    
    score = rmse(model, test)
    return score if not (np.isnan(score) or np.isinf(score)) else 999

### 1. Manual Baseline

In [ ]:
print("Running Manual Baseline...")
start = time.time()
model = MF(n_users, n_items, k=40, learning_rate=0.01, reg_param=0.08, epochs=10)
model.train(train)
score = rmse(model, test)
elapsed = time.time() - start

results_list.append({"Model": "Manual", "RMSE": score, "Time": elapsed})
print(f"Manual RMSE: {score:.4f}")

### 2. Grid Search (Small Scale)

In [ ]:
print("Running Grid Search...")
k_list = [20, 50]
lr_list = [0.005, 0.01]
reg_list = [0.05, 0.1]

best_score = 999
start = time.time()

for k, lr, reg in itertools.product(k_list, lr_list, reg_list):
    model = MF(n_users, n_items, k, lr, reg, epochs=5)
    model.train(train)
    score = rmse(model, test)
    if score < best_score:
        best_score = score

elapsed = time.time() - start
results_list.append({"Model": "Grid Search", "RMSE": best_score, "Time": elapsed})
print(f"Grid Search RMSE: {best_score:.4f}")

### 3. PSO Optimized

In [ ]:
print("\nRunning PSO Optimization...")
lb = [0.002, 0.02, 20, 5]
ub = [0.02, 0.2, 80, 20]

pso = ParticleSwarmOptimization(
    fitness_func=fitness_function,
    n_particles=5,
    n_iter=10,
    lower_bound=lb,
    upper_bound=ub
)

start = time.time()
pso_best_params, pso_best_rmse, pso_history = pso.optimize()
elapsed = time.time() - start

results_list.append({"Model": "PSO-MF", "RMSE": pso_best_rmse, "Time": elapsed})
print(f"PSO RMSE: {pso_best_rmse:.4f}")

### 4. WOA Optimized (Load from 01)

In [ ]:
if os.path.exists(WO_RESULTS_PATH):
    with open(WO_RESULTS_PATH, "r") as f:
        woa_data = json.load(f)
    
    results_list.append({"Model": "WOA-MF", "RMSE": woa_data["rmse"], "Time": woa_data["time"]})
    print(f"\nLoaded WOA RMSE: {woa_data['rmse']:.4f}")
else:
    print("WOA results not found. Please run 01_train_woa.ipynb first.")

In [ ]:
final_df = pd.DataFrame(results_list)
print(f"\nSaving final results to: {FINAL_RESULTS_PATH}")
final_df.to_csv(FINAL_RESULTS_PATH, index=False)
final_df